# Notebook 17 — Tiny Self-Supervised Model (Dummy Foundation Model Demo)
Goal: Understand the **steps** involved in training a foundation-model-like system without needing large data or GPUs.

This notebook does **not** build a real foundation model. Instead, it demonstrates the core training pattern on a tiny toy example:

`raw inputs → self-supervised objective → encoder → embedding space`

We will train a small neural network to predict whether two augmented views came from the same original item. This is a simplified version of the representation-learning idea behind many foundation models.

## Section 1 — Import Libraries

In [ ]:
# If needed:
# pip install torch scikit-learn matplotlib numpy pandas

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

## Section 2 — High-Level Idea
A real foundation model is usually trained with a **self-supervised objective** such as:

- masked token prediction
- masked patch prediction
- contrastive learning
- next-token prediction

Here we will use a tiny contrastive-style idea:

1. start with some original examples
2. make two noisy versions of each example
3. train a model so embeddings from the same source are close
4. embeddings from different sources should be less similar

This is only a toy demonstration, but the workflow is real.

## Section 3 — Create Tiny Toy Data

In [ ]:
np.random.seed(42)

# Create 3 underlying groups in 2D space
group1 = np.random.normal(loc=[-2, 0], scale=0.3, size=(20, 2))
group2 = np.random.normal(loc=[ 2, 0], scale=0.3, size=(20, 2))
group3 = np.random.normal(loc=[ 0, 2], scale=0.3, size=(20, 2))

base_data = np.vstack([group1, group2, group3])

plt.figure(figsize=(5, 5))
plt.scatter(base_data[:,0], base_data[:,1])
plt.title("Underlying Toy Data")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

base_data.shape

## Section 4 — Create Two Augmented Views
In self-supervised learning, we often create different views of the same item.
Here, each view is just the original point plus a little random noise.

In [ ]:
def augment(x, noise_scale=0.15):
    noise = np.random.normal(scale=noise_scale, size=x.shape)
    return x + noise

view1 = augment(base_data)
view2 = augment(base_data)

plt.figure(figsize=(5, 5))
plt.scatter(view1[:,0], view1[:,1], label="view1", alpha=0.6)
plt.scatter(view2[:,0], view2[:,1], label="view2", alpha=0.6)
plt.legend()
plt.title("Two Augmented Views")
plt.show()

## Section 5 — Build Positive and Negative Pairs
We need a training signal.

- Positive pair: two views of the same original point
- Negative pair: views from different original points

We will train a small model to tell whether a pair is positive or negative.

In [ ]:
# Positive pairs
positive_pairs = [(view1[i], view2[i], 1) for i in range(len(base_data))]

# Negative pairs
rng = np.random.default_rng(42)
negative_pairs = []
for i in range(len(base_data)):
    j = rng.integers(0, len(base_data))
    while j == i:
        j = rng.integers(0, len(base_data))
    negative_pairs.append((view1[i], view2[j], 0))

pairs = positive_pairs + negative_pairs
rng.shuffle(pairs)

X_left = np.array([p[0] for p in pairs], dtype=np.float32)
X_right = np.array([p[1] for p in pairs], dtype=np.float32)
y = np.array([p[2] for p in pairs], dtype=np.float32)

X_left.shape, X_right.shape, y.shape

## Section 6 — Define a Tiny Encoder
This encoder turns each input into an embedding vector.

In a real foundation model, the encoder might be:

- a transformer
- a vision transformer
- a protein language model

Here it is just a tiny MLP.

In [ ]:
class TinyEncoder(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=16, embedding_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )

    def forward(self, x):
        return self.net(x)


class PairClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x1, x2):
        z1 = self.encoder(x1)
        z2 = self.encoder(x2)
        combined = torch.cat([z1, z2], dim=1)
        logits = self.classifier(combined)
        return logits, z1, z2


encoder = TinyEncoder()
model = PairClassifier(encoder)
model

## Section 7 — Convert Data to Tensors

In [ ]:
X_left_t = torch.tensor(X_left)
X_right_t = torch.tensor(X_right)
y_t = torch.tensor(y).unsqueeze(1)

X_left_t.shape, y_t.shape

## Section 8 — Train the Model
We use a binary objective: predict whether two views come from the same source.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []

for epoch in range(300):
    optimizer.zero_grad()
    logits, _, _ = model(X_left_t, X_right_t)
    loss = criterion(logits, y_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

losses[-1]

## Section 9 — Extract Learned Embeddings
Now we use the trained encoder by itself.
This is the key idea in applied foundation-model workflows:

- train encoder with a self-supervised objective
- discard the training head
- keep the encoder
- use embeddings downstream

In [ ]:
with torch.no_grad():
    learned_embeddings = encoder(torch.tensor(base_data, dtype=torch.float32)).numpy()

learned_embeddings.shape

## Section 10 — Visualize the Embedding Space

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(learned_embeddings)

plt.figure(figsize=(5, 5))
plt.scatter(coords[:,0], coords[:,1])
plt.title("PCA of Learned Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Section 11 — Compare Embedding Similarity
Points that came from similar underlying groups should often have more similar embeddings.

In [ ]:
sim = cosine_similarity(learned_embeddings[:10])
np.round(sim, 3)

## Section 12 — What This Corresponds To in a Real Foundation Model
This tiny demo maps onto real workflows like this:

| Toy notebook step | Real foundation model analogue |
|---|---|
| small encoder | transformer / ViT / language model |
| noisy views | text masking / image augmentation / patch masking |
| pair objective | contrastive / masked / self-supervised objective |
| learned embeddings | pretrained representation space |
| downstream use | classification / clustering / retrieval / prediction |

The notebook is not building a real foundation model, but it demonstrates the training logic.

## Section 13 — Exercises

1. Change the embedding dimension from 8 to 4 or 16 and retrain.
2. Increase the augmentation noise and see what happens to training.
3. Reduce the number of training epochs and compare results.
4. Plot the original data and learned embedding PCA side by side.
5. Replace the pair-classification objective with a different toy objective if you want to experiment.

## Skills Gained
- understanding the training *idea* behind foundation models
- seeing how self-supervised learning creates embeddings
- separating encoder training from downstream use
- understanding why pretrained embeddings are useful

After this notebook, the earlier multimodal notebooks should make more intuitive sense: they use pretrained encoders that were trained in a much larger version of this general pattern.